# Publisher Analysis: Gaming Platforms 2025
**Task #5 — Yayıncı Analizi (Publisher Analysis)**

## Purpose
Analyze publisher dominance across Steam, PlayStation, and Xbox platforms using Gold-layer tables built in BigQuery. This notebook handles statistical testing, machine learning clustering, and professional visualizations for the final presentation.

## Data Sources
7 Gold-layer tables exported from BigQuery, built from 131,884 games across 3 platforms.

| Table | What It Answers | Rows |
|-------|-----------------|------|
| gold_market_share | Who dominates which platform? | 54,668 |
| gold_platform_strategy | Single vs multi-platform impact? | 51,192 |
| gold_pricing_strategy | Budget vs mid vs premium clusters? | 45,154 |
| gold_genre_specialization | Which genres does each publisher own? | 161,112 |
| gold_indie_vs_major | How do 49K indies compare to 105 majors? | 4 |
| gold_top20_dashboard | Complete profile of 20 largest publishers | 20 |
| gold_release_patterns | How has the market changed 2015-2024? | 77,850 |

## Tools
- **Pandas** — data loading and manipulation
- **Plotly** — interactive, publication-quality charts
- **scipy.stats** — hypothesis testing (t-test)
- **scikit-learn** — K-means clustering for publisher segmentation

**Author:** Poi | **Date:** 2026-03-04

## 1. Setup & Imports

Colab has Pandas and Plotly pre-installed. We import everything we need upfront.

**Python concept — `import`:** Loading external libraries so we can use their functions. `pd` and `px` are short aliases so we don't type the full name every time.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Plotly default template for consistent styling
import plotly.io as pio
pio.templates.default = 'plotly_white'

print('All libraries loaded successfully.')

## 2. Load Gold Layer CSVs

**Python concept — `pd.read_csv()`:** Pandas reads a CSV file and creates a DataFrame — think of it as a SQL table in Python. Each column has a name, each row has an index number.

Upload your 7 renamed CSV files to Colab first (drag into the Files panel on the left), then run this cell.

In [ ]:
# Upload files to Colab first, then load them
from google.colab import files

# If files are already uploaded to Colab's /content/ directory:
market_share = pd.read_csv('gold_market_share.csv')
platform_strategy = pd.read_csv('gold_platform_strategy.csv')
pricing_strategy = pd.read_csv('gold_pricing_strategy.csv')
genre_spec = pd.read_csv('gold_genre_specialization.csv')
indie_major = pd.read_csv('gold_indie_vs_major.csv')
top20 = pd.read_csv('gold_top20_dashboard.csv')
release_patterns = pd.read_csv('gold_release_patterns.csv')

print('All 7 Gold tables loaded.')
print(f'  market_share:      {market_share.shape[0]:>7,} rows × {market_share.shape[1]} cols')
print(f'  platform_strategy: {platform_strategy.shape[0]:>7,} rows × {platform_strategy.shape[1]} cols')
print(f'  pricing_strategy:  {pricing_strategy.shape[0]:>7,} rows × {pricing_strategy.shape[1]} cols')
print(f'  genre_spec:        {genre_spec.shape[0]:>7,} rows × {genre_spec.shape[1]} cols')
print(f'  indie_major:       {indie_major.shape[0]:>7,} rows × {indie_major.shape[1]} cols')
print(f'  top20:             {top20.shape[0]:>7,} rows × {top20.shape[1]} cols')
print(f'  release_patterns:  {release_patterns.shape[0]:>7,} rows × {release_patterns.shape[1]} cols')

## 3. Data Overview

**Python concept — `.head()` and `.dtypes`:** `.head()` shows the first 5 rows (like `LIMIT 5` in SQL). `.dtypes` shows column types (like `INFORMATION_SCHEMA` in BigQuery).

Let's verify our data loaded correctly.

In [ ]:
# Quick sanity check on each table
print('=== Top 20 Dashboard (our star table) ===')
top20.head()

In [ ]:
# Check indie vs major — this is our 4-row summary
print('=== Indie vs Major ===')
indie_major

## 4. Chart 1 — Market Share by Platform

**Question:** Who dominates each platform, and by how much?

**Why this chart:** A grouped bar chart lets us compare the top publishers side-by-side across platforms. We'll take the top 10 publishers per platform to keep it readable.

**Python concept — filtering DataFrames:** `df[df['column'] == value]` is like `WHERE column = value` in SQL. We chain conditions with `&` (AND) and `|` (OR).

In [ ]:
# Top 10 publishers per platform (excluding 'unknown')
top10_per_platform = (
    market_share[market_share['publisher'] != 'unknown']
    .query('platform_rank <= 10')
    .sort_values(['platform_group', 'platform_rank'])
)

fig1 = px.bar(
    top10_per_platform,
    x='publisher',
    y='market_share_pct',
    color='platform_group',
    barmode='group',
    title='Top 10 Publishers by Market Share — Per Platform',
    labels={
        'market_share_pct': 'Market Share (%)',
        'publisher': 'Publisher',
        'platform_group': 'Platform'
    },
    color_discrete_map={'PS': '#003087', 'Steam': '#1B2838', 'Xbox': '#107C10'},
    height=500
)

fig1.update_layout(
    xaxis_tickangle=-45,
    legend_title='Platform',
    font=dict(size=12)
)
fig1.show()

**Insight:** No single publisher dominates any platform. Even the #1 known publisher on Steam (Big Fish Games) holds only 0.52%. PlayStation's top publishers are budget/trophy publishers, not AAA names. Xbox is the only platform where traditional publishers (Microsoft, EA) appear prominently.

## 5. Chart 2 — Platform Strategy: Single vs Multi-Platform

**Question:** Do multi-platform publishers outperform single-platform ones?

**Why this chart:** We need to compare averages across 3 strategy groups (single/dual/tri-platform) on multiple metrics. A grouped bar chart with facets works well here.

**Python concept — `.groupby().agg()`:** This is the Pandas equivalent of `GROUP BY` with aggregate functions in SQL.

In [ ]:
# Aggregate by platform strategy
strategy_summary = (
    platform_strategy
    .groupby('platform_strategy')
    .agg(
        publisher_count=('publisher', 'count'),
        avg_games=('game_count', 'mean'),
        avg_price=('avg_usd', 'mean'),
        avg_genres=('genre_count', 'mean')
    )
    .round(2)
    .reset_index()
)

# Reorder for logical display
strategy_order = ['single-platform', 'dual-platform', 'tri-platform']
strategy_summary['platform_strategy'] = pd.Categorical(
    strategy_summary['platform_strategy'],
    categories=strategy_order,
    ordered=True
)
strategy_summary = strategy_summary.sort_values('platform_strategy')

print(strategy_summary.to_string(index=False))

In [ ]:
# Multi-metric comparison chart
fig2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Avg Games per Publisher', 'Avg Price (USD)', 'Avg Genre Diversity'),
    horizontal_spacing=0.12
)

colors = {'single-platform': '#636EFA', 'dual-platform': '#EF553B', 'tri-platform': '#00CC96'}

for i, (metric, title) in enumerate([
    ('avg_games', 'Avg Games'),
    ('avg_price', 'Avg Price ($)'),
    ('avg_genres', 'Avg Genres')
], 1):
    for _, row in strategy_summary.iterrows():
        fig2.add_trace(
            go.Bar(
                x=[row['platform_strategy']],
                y=[row[metric]],
                name=row['platform_strategy'],
                marker_color=colors[row['platform_strategy']],
                showlegend=(i == 1),
                text=[row[metric]],
                textposition='outside'
            ),
            row=1, col=i
        )

fig2.update_layout(
    title='Multi-Platform Publishers Outperform on Every Metric',
    height=450,
    font=dict(size=12),
    showlegend=True
)
fig2.show()

**Insight:** Tri-platform publishers (only 2% of all publishers) produce ~15x more games, charge ~2x higher prices, and span ~7x more genres than single-platform publishers. Multi-platform reach correlates with every success metric.

## 6. Chart 3 — Pricing Strategy by Platform

**Question:** How does pricing differ across platforms?

**Why this chart:** A stacked bar showing tier distribution per platform reveals each platform's pricing personality.

**Python concept — `pivot_table()`:** Reshapes data from long format to wide format — like a SQL `PIVOT`. We need this to get platforms as rows and price tiers as columns.

In [ ]:
# Aggregate tier counts per platform
platform_tiers = (
    pricing_strategy
    .groupby('platform_group')
    .agg(
        budget=('budget_count', 'sum'),
        mid=('mid_count', 'sum'),
        premium=('premium_count', 'sum')
    )
    .reset_index()
)

# Calculate percentages
for col in ['budget', 'mid', 'premium']:
    platform_tiers[f'{col}_pct'] = round(
        platform_tiers[col] / platform_tiers[['budget', 'mid', 'premium']].sum(axis=1) * 100, 1
    )

print(platform_tiers[['platform_group', 'budget_pct', 'mid_pct', 'premium_pct']].to_string(index=False))

In [ ]:
# Stacked percentage bar chart
fig3 = go.Figure()

tier_colors = {'Budget (<$10)': '#636EFA', 'Mid ($10-30)': '#EF553B', 'Premium (>$30)': '#FFA15A'}

for tier, col, color in [
    ('Budget (<$10)', 'budget_pct', '#636EFA'),
    ('Mid ($10-30)', 'mid_pct', '#EF553B'),
    ('Premium (>$30)', 'premium_pct', '#FFA15A')
]:
    fig3.add_trace(go.Bar(
        x=platform_tiers['platform_group'],
        y=platform_tiers[col],
        name=tier,
        marker_color=color,
        text=platform_tiers[col].apply(lambda x: f'{x}%'),
        textposition='inside'
    ))

fig3.update_layout(
    barmode='stack',
    title='Price Tier Distribution by Platform — Each Platform Has a Pricing Personality',
    yaxis_title='Percentage of Games (%)',
    xaxis_title='Platform',
    height=450,
    font=dict(size=12)
)
fig3.show()

**Insight:** Steam is overwhelmingly budget-dominated. Xbox leans mid-tier. PlayStation has a unique volume-budget model. Premium games are under 5% everywhere, but Steam premium titles average $73 (likely simulation/professional software) vs ~$42 on consoles.

## 7. Chart 4 — Genre Specialization Heatmap

**Question:** What genres do the top publishers specialize in?

**Why a heatmap:** It shows two dimensions (publisher × genre) with color intensity representing concentration. Perfect for spotting specialists vs generalists at a glance.

**Python concept — `.pivot()`:** Turns long data (publisher, genre, value) into a matrix (publishers as rows, genres as columns, values as cells).

In [ ]:
# Get top 15 publishers from our top20 dashboard
top_publishers = top20['publisher'].head(15).tolist()

# Filter genre data to these publishers, top 3 genres each
heatmap_data = (
    genre_spec[
        (genre_spec['publisher'].isin(top_publishers)) &
        (genre_spec['genre_rank'] <= 5)
    ]
    [['publisher', 'genre', 'genre_share_pct']]
)

# Find the most common genres across these publishers
top_genres = heatmap_data['genre'].value_counts().head(12).index.tolist()
heatmap_filtered = heatmap_data[heatmap_data['genre'].isin(top_genres)]

# Pivot to matrix form
heatmap_matrix = heatmap_filtered.pivot(
    index='publisher',
    columns='genre',
    values='genre_share_pct'
).fillna(0)

# Sort publishers by game count (using top20 order)
ordered_publishers = [p for p in top_publishers if p in heatmap_matrix.index]
heatmap_matrix = heatmap_matrix.loc[ordered_publishers]

fig4 = px.imshow(
    heatmap_matrix,
    text_auto='.1f',
    color_continuous_scale='Blues',
    title='Genre Specialization — Top 15 Publishers × Top Genres',
    labels={'color': 'Genre Share (%)'},
    height=600,
    aspect='auto'
)

fig4.update_layout(
    xaxis_title='Genre',
    yaxis_title='Publisher',
    font=dict(size=11)
)
fig4.show()

**Insight:** Volume publishers (Eastasiasoft, Ratalaika) concentrate in Platformer and Action. AAA publishers (EA in Sports, Ubisoft in Action) have clear specialties but broader spread. THIGames is the most concentrated — 53% Action.

## 8. Chart 5 — Indie vs Major: The Structural Divide

**Question:** How different are indie and major publishers across key metrics?

**Why this chart:** A clean comparison table visualization + bar chart shows the scale of the gap. We already have this pre-aggregated from BigQuery.

In [ ]:
# Display the indie vs major comparison
display_cols = ['publisher_tier', 'publisher_count', 'avg_games_per_publisher',
                'avg_price_usd', 'avg_platform_count', 'pct_single_platform']

print(indie_major[display_cols].to_string(index=False))

In [ ]:
# Visual comparison — key metrics side by side
comparison_metrics = ['avg_games_per_publisher', 'avg_price_usd', 'avg_platform_count']
metric_labels = ['Avg Games/Publisher', 'Avg Price (USD)', 'Avg Platforms']

# Filter to just major and indie for clean comparison
mi = indie_major[indie_major['publisher_tier'].isin(['major', 'indie'])].copy()

fig5 = make_subplots(
    rows=1, cols=3,
    subplot_titles=metric_labels,
    horizontal_spacing=0.12
)

tier_colors_im = {'major': '#EF553B', 'indie': '#636EFA'}

for i, metric in enumerate(comparison_metrics, 1):
    for _, row in mi.iterrows():
        fig5.add_trace(
            go.Bar(
                x=[row['publisher_tier']],
                y=[row[metric]],
                name=row['publisher_tier'],
                marker_color=tier_colors_im[row['publisher_tier']],
                showlegend=(i == 1),
                text=[f"{row[metric]:,.1f}"],
                textposition='outside'
            ),
            row=1, col=i
        )

fig5.update_layout(
    title='The Structural Divide: 105 Majors vs 49,834 Indies',
    height=400,
    font=dict(size=12)
)
fig5.show()

**Insight:** The gap is structural, not gradual. Major publishers average 235 games vs 1.5 for indies. 96.6% of indie publishers are locked to a single platform. The price gap ($15.70 vs $7.91) reflects fundamentally different business models.

## 9. Chart 6 — The Scissors Pattern (2015-2024)

**Question:** How has the indie vs major divide evolved over time?

**Why a line chart:** Time-series data needs line charts to show trends. The "scissors" pattern — where indie volume explodes while prices stay flat, and major prices rise while volume grows steadily — is the most striking finding in the dataset.

**Python concept — `.groupby()` with multiple aggregations:** Same as SQL `GROUP BY` with multiple aggregate functions.

In [ ]:
# Aggregate by tier and year (2015-2024 for clearest trend)
yearly_trends = (
    release_patterns[
        (release_patterns['release_year'].between(2015, 2024)) &
        (release_patterns['publisher_tier'].isin(['major', 'indie']))
    ]
    .groupby(['release_year', 'publisher_tier'])
    .agg(
        total_games=('games_released', 'sum'),
        avg_price=('avg_usd_that_year', 'mean'),
        publisher_count=('publisher', 'nunique')
    )
    .round(2)
    .reset_index()
)

yearly_trends.head(10)

In [ ]:
# Scissors chart: dual-axis — volume + price over time
fig6 = make_subplots(specs=[[{'secondary_y': True}]])

for tier, color, dash in [('indie', '#636EFA', 'solid'), ('major', '#EF553B', 'solid')]:
    tier_data = yearly_trends[yearly_trends['publisher_tier'] == tier]

    # Volume line (left axis)
    fig6.add_trace(
        go.Scatter(
            x=tier_data['release_year'],
            y=tier_data['total_games'],
            name=f'{tier.title()} — Games Released',
            line=dict(color=color, width=3),
            mode='lines+markers'
        ),
        secondary_y=False
    )

    # Price line (right axis)
    fig6.add_trace(
        go.Scatter(
            x=tier_data['release_year'],
            y=tier_data['avg_price'],
            name=f'{tier.title()} — Avg Price ($)',
            line=dict(color=color, width=2, dash='dash'),
            mode='lines+markers'
        ),
        secondary_y=True
    )

fig6.update_layout(
    title='The Scissors Pattern: Indie Volume Explodes, Major Prices Rise (2015-2024)',
    height=500,
    font=dict(size=12),
    legend=dict(x=0.01, y=0.99)
)
fig6.update_yaxes(title_text='Games Released', secondary_y=False)
fig6.update_yaxes(title_text='Avg Price (USD)', secondary_y=True)
fig6.update_xaxes(title_text='Year', dtick=1)
fig6.show()

**Insight:** The scissors pattern is clear. From 2015-2024, indie publisher count grew 8.5x while prices stayed flat around $8. Major publishers grew output 5.7x AND raised prices 46% ($15.67→$22.88). The price gap doubled from $7.47 to $14.54. The 2021 inflection point — where major output nearly doubled in one year — likely reflects pandemic-era release backlogs and digital distribution acceleration.

## 10. Chart 7 — Top 20 Publishers Dashboard

**Question:** What does the complete profile of the largest publishers look like?

**Why a horizontal bar with annotations:** Shows ranking clearly with key stats visible at a glance. This is the presentation centerpiece.

In [ ]:
# Top 20 publishers — game count with platform breakdown
fig7 = go.Figure()

# Stacked horizontal bars by platform
for platform, col, color in [
    ('PlayStation', 'ps_games', '#003087'),
    ('Steam', 'steam_games', '#1B2838'),
    ('Xbox', 'xbox_games', '#107C10')
]:
    fig7.add_trace(go.Bar(
        y=top20['publisher'],
        x=top20[col],
        name=platform,
        orientation='h',
        marker_color=color
    ))

fig7.update_layout(
    barmode='stack',
    title='Top 20 Publishers by Game Count — Platform Breakdown',
    xaxis_title='Number of Games',
    yaxis=dict(autorange='reversed'),  # Largest at top
    height=600,
    font=dict(size=12),
    legend=dict(x=0.7, y=0.05)
)
fig7.show()

**Insight:** The two business models are visible in the bars. Volume publishers (Eastasiasoft, Ratalaika, THIGames) are PS-heavy with massive game counts. AAA publishers (EA, Ubisoft, Sega) show balanced tri-platform distribution. Eastasiasoft (1,356 games) is bigger than EA (684) by raw volume.

## 11. Hypothesis Test — Major vs Indie Pricing

**Research question:** Is the average price difference between major and indie publishers statistically significant, or could it be due to random variation?

**Python concept — scipy t-test:** A t-test compares the means of two groups and tells you the probability (p-value) that the observed difference happened by chance.
- **H₀ (null hypothesis):** Major and indie publishers have the same average price.
- **H₁ (alternative hypothesis):** Major publishers charge significantly more than indie publishers.
- **Significance level:** α = 0.05 (standard threshold — if p < 0.05, we reject H₀).

**Why we use publisher-level avg_usd:** Each publisher's average price is one observation. We're comparing 105 major publisher averages vs 49,834 indie publisher averages.

In [ ]:
# Classify publishers into tiers using platform_strategy data
# (has game_count and avg_usd per publisher)
ps = platform_strategy.copy()
ps['publisher_tier'] = ps['game_count'].apply(
    lambda x: 'major' if x >= 100 else ('mid-tier' if x >= 10 else 'indie')
)

# Extract price arrays for major and indie
major_prices = ps[ps['publisher_tier'] == 'major']['avg_usd'].dropna()
indie_prices = ps[ps['publisher_tier'] == 'indie']['avg_usd'].dropna()

print(f'Major publishers: n={len(major_prices)}, mean=${major_prices.mean():.2f}, std=${major_prices.std():.2f}')
print(f'Indie publishers: n={len(indie_prices)}, mean=${indie_prices.mean():.2f}, std=${indie_prices.std():.2f}')
print(f'Price difference: ${major_prices.mean() - indie_prices.mean():.2f}')

In [ ]:
# Welch's t-test (doesn't assume equal variances — appropriate here
# because we have 105 majors vs ~49K indies with different spreads)
t_stat, p_value = stats.ttest_ind(major_prices, indie_prices, equal_var=False)

print(f'Welch\'s t-test results:')
print(f'  t-statistic: {t_stat:.4f}')
print(f'  p-value:     {p_value:.2e}')
print()

if p_value < 0.05:
    print(f'✅ REJECT H₀ (p < 0.05)')
    print(f'The price difference between major and indie publishers')
    print(f'is statistically significant — not due to random chance.')
else:
    print(f'❌ FAIL TO REJECT H₀ (p >= 0.05)')
    print(f'Cannot conclude that the price difference is significant.')

In [ ]:
# Visualize the price distributions
fig_ttest = go.Figure()

fig_ttest.add_trace(go.Histogram(
    x=indie_prices,
    name=f'Indie (n={len(indie_prices):,})',
    marker_color='#636EFA',
    opacity=0.7,
    nbinsx=50
))

fig_ttest.add_trace(go.Histogram(
    x=major_prices,
    name=f'Major (n={len(major_prices)})',
    marker_color='#EF553B',
    opacity=0.7,
    nbinsx=30
))

fig_ttest.update_layout(
    barmode='overlay',
    title=f'Price Distribution: Major vs Indie Publishers (p={p_value:.2e})',
    xaxis_title='Average Price (USD)',
    yaxis_title='Number of Publishers',
    height=450,
    font=dict(size=12)
)
fig_ttest.show()

**Interpretation:** With a p-value far below 0.05, we confidently reject the null hypothesis. The ~$8 price gap between major and indie publishers is not random — it reflects fundamentally different business models. Major publishers price for margin; indie publishers price for accessibility.

## 12. K-Means Clustering — Publisher Segmentation

**Question:** Can we discover natural publisher groupings beyond our manual tier definitions?

**Python concept — K-Means:** An algorithm that groups data points into K clusters based on similarity. It works by:
1. Placing K random center points
2. Assigning each data point to the nearest center
3. Moving centers to the middle of their assigned points
4. Repeating until stable

**Why StandardScaler:** K-Means uses distance, so features with larger numbers (game_count: 1-1356) would dominate features with smaller numbers (platform_count: 1-3). Scaling puts everything on the same scale.

**Features:** game_count, avg_usd, platform_count, genre_count

In [ ]:
# Prepare clustering data — publishers with complete info
cluster_data = platform_strategy[['publisher', 'game_count', 'avg_usd', 'platform_count', 'genre_count']].copy()
cluster_data = cluster_data.dropna()

# Features for clustering
features = ['game_count', 'avg_usd', 'platform_count', 'genre_count']
X = cluster_data[features].values

# Scale features so K-Means treats them equally
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Clustering {len(X):,} publishers on {len(features)} features')
print(f'Features: {features}')

In [ ]:
# Find optimal K using the Elbow Method
# Try K=2 through K=8, measure how tight the clusters are (inertia)
inertias = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig_elbow = px.line(
    x=list(K_range), y=inertias,
    markers=True,
    title='Elbow Method — Finding Optimal Number of Clusters',
    labels={'x': 'Number of Clusters (K)', 'y': 'Inertia (lower = tighter clusters)'},
    height=400
)
fig_elbow.show()

**Reading the elbow chart:** Look for where the line bends sharply — adding more clusters after that point gives diminishing returns. We expect K=3 or K=4 to be the elbow, matching our intuition about indie/mid/major (or a fourth 'volume publisher' segment).

In [ ]:
# Apply K-Means with chosen K
# Using K=4 — we expect: indie, mid-tier, AAA, volume publisher segments
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_data['cluster'] = kmeans.fit_predict(X_scaled)

# Analyze cluster profiles
cluster_profiles = (
    cluster_data
    .groupby('cluster')
    .agg(
        count=('publisher', 'count'),
        avg_games=('game_count', 'mean'),
        avg_price=('avg_usd', 'mean'),
        avg_platforms=('platform_count', 'mean'),
        avg_genres=('genre_count', 'mean')
    )
    .round(2)
)

# Label clusters based on their profiles
print('Cluster Profiles:')
print(cluster_profiles.to_string())
print()
print('Sample publishers per cluster:')
for c in sorted(cluster_data['cluster'].unique()):
    samples = cluster_data[cluster_data['cluster'] == c].nlargest(5, 'game_count')['publisher'].tolist()
    print(f'  Cluster {c}: {samples}')

In [ ]:
# Visualize clusters — game count vs avg price, colored by cluster
fig_km = px.scatter(
    cluster_data,
    x='game_count',
    y='avg_usd',
    color='cluster',
    hover_data=['publisher', 'platform_count', 'genre_count'],
    title='K-Means Publisher Clusters: Game Volume vs Average Price',
    labels={
        'game_count': 'Number of Games',
        'avg_usd': 'Average Price (USD)',
        'cluster': 'Cluster'
    },
    height=500,
    color_continuous_scale='Viridis'
)

# Annotate notable publishers
notable = ['eastasiasoft', 'electronic arts', 'ubisoft', 'ratalaika games']
for pub in notable:
    row = cluster_data[cluster_data['publisher'] == pub]
    if not row.empty:
        fig_km.add_annotation(
            x=row['game_count'].values[0],
            y=row['avg_usd'].values[0],
            text=pub.title(),
            showarrow=True,
            arrowhead=2,
            font=dict(size=10)
        )

fig_km.show()

**Interpretation:** K-Means confirms the two business models we identified in SQL: the AAA cluster (fewer games, higher prices, multi-platform) and the Volume cluster (many games, budget prices). The algorithm also separates the massive indie tail as its own cluster — validating our tier definitions with an independent, data-driven method.

## 13. Export Charts as PNG

Export key charts for the README and Google Slides presentation.

In [ ]:
# Install kaleido for static image export in Colab
!pip install -q plotly --upgrade
!pip install -q kaleido==0.2.1

# Export key charts
charts = {
    'market_share_top10.png': fig1,
    'platform_strategy_comparison.png': fig2,
    'pricing_by_platform.png': fig3,
    'genre_heatmap.png': fig4,
    'indie_vs_major.png': fig5,
    'scissors_pattern.png': fig6,
    'top20_dashboard.png': fig7,
    'price_distribution_ttest.png': fig_ttest,
    'kmeans_clusters.png': fig_km,
    'elbow_method.png': fig_elbow,
}

for name, fig in charts.items():
    fig.write_image(name, scale=2)  # 2x resolution for crisp images
    print(f'Saved: {name}')

print(f'\n{len(charts)} charts exported as PNG.')

In [ ]:
# Download all PNGs
from google.colab import files
import glob

for png in glob.glob('*.png'):
    files.download(png)
    print(f'Downloaded: {png}')

## 14. Summary of Findings

### Statistical Results
- **T-test:** Major publishers charge significantly more than indie publishers (p < 0.001). The ~$8 price gap is not random — it reflects different business models.
- **K-Means:** Clustering confirms two distinct publisher archetypes (AAA vs Volume) plus a massive indie tail, validating our SQL-defined tiers with machine learning.

### Key Story for Presentation
1. **No monopoly exists** — gaming publishing is radically fragmented across all platforms
2. **Two winning models** — AAA (premium, multi-platform, genre-diverse) vs Volume (budget, platform-focused, genre-concentrated)
3. **The indie trap** — 96.6% of indie publishers are locked to one platform with minimal pricing power
4. **The scissors pattern** — indie volume explodes while prices stagnate; major prices rise while output grows. The gap is widening.
5. **Platform personalities** — Steam = budget paradise, Xbox = mid-tier territory, PS = volume-budget with unique trophy publisher ecosystem

### Charts Exported
10 publication-ready PNG files for README and Google Slides.

---
*Notebook by Poi | Gaming Publisher Analysis | March 2026*